# Этап 3: компактный pair-reranker поверх OSNet

Запустить Run All в существующем `.venv`. Обучается только маленькая голова, OSNet и MVP не меняются.
Внутренний отбор использует отдельный OSNet, не обучавшийся на 184 inner-validation identity.
Рецепт переносится на признаки MVP / 925 identity с обучением новой головы с нуля.

Описание loss, признаков, splits, ограничений и артефактов: [README.md](README.md).

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/pair_reranker_experiment.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук из Car-classification-MSK')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from training.pair_reranker_experiment import EXPERIMENT, run
OUTPUT = EXPERIMENT / 'results/run_01'
print('Python:', sys.executable)
print('Результаты:', OUTPUT)
print('CPU, 2 потока; обучения большого encoder нет. Один экземпляр одновременно.')

## Автоматический эксперимент

Проверка provenance → экспорт/признаки чистого inner OSNet → обучение 2 голов (≤25 эпох) → выбор по inner → новая голова на MVP-признаках → calibration → фиксация → validation, масочный аудит и отчёт.

Кеш сохраняется после batch, обучение — после эпохи. При прерывании достаточно Run All; готовый запуск только проверяется и читается. Не удаляйте старые JSON. Для изменённых настроек используйте новую OUTPUT.

In [ ]:
report = run(OUTPUT)

## Итог

Сравниваем отдельно ранжирование и отказ. Ни результаты validation, ни ручные маски не используются для повторного подбора. MVP не заменяется автоматически.

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'RESULTS.md').read_text(encoding='utf-8')))